In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Crab: DC4 data preparation

Prepare multienergy observed and simulated-background histograms. Run and inspect each cell in order. This notebook does not perform a fit. The simulated background is provisional; its provenance, normalization and temporal compatibility must be evaluated later.

## Imports and paths

Locate the workflow and import its existing path definitions.

In [3]:
from pathlib import Path
import sys

# Locate the workflow from the kernel working directory.
working_directory = Path.cwd().resolve()
candidates = [working_directory, *working_directory.parents]
workflow_dir = next(
    (directory for directory in candidates if (directory / "src" / "path_to_files.py").is_file()),
    None,
)
if workflow_dir is None and (working_directory / "workflow" / "src" / "path_to_files.py").is_file():
    workflow_dir = working_directory / "workflow"
if workflow_dir is None:
    raise FileNotFoundError("Start the notebook from the repository or workflow directory.")

# Import COSIpy and the project's existing paths.
if str(workflow_dir) not in sys.path:
    sys.path.insert(0, str(workflow_dir))
import cosipy as cp
from cosipy.util import fetch_wasabi_file
from src import path_to_files as paths

import numpy as np
import astropy.units as u
from astropy.io import fits


print("Working directory:", working_directory)
print("Workflow directory:", workflow_dir)

Working directory: /Users/maura/Documents/COSI/data-challenge-4/workflow/notebooks
Workflow directory: /Users/maura/Documents/COSI/data-challenge-4/workflow


## Input and output locations

The observations contain all selected events, not only Crab photons. Separate output names preserve the 511 keV products.

In [4]:
# Define shared inputs and the continuum binning configuration.
weekly_paths = paths.weekly_paths
background_path = paths.simulated_background_path
orientation_path = paths.data_ori_path
response_path = paths.data_responsecon_path
config_path = paths.config_continuum_path

# Define independent multienergy output paths.
output_dir = paths.data_dir_out
weekly_binned_paths = {
    week: output_dir / f"dc4_week_{week}_crab_continuum_binned.hdf5"
    for week in weekly_paths
}
background_binned_path = output_dir / "simulated_background_crab_continuum_binned.hdf5"

print("Configuration:", config_path)
print("Orientation:", orientation_path)
print("Continuum response:", response_path)
print("Simulated background:", background_path)
print("Weekly files:", len(weekly_paths))
print("Output directory:", output_dir)

Configuration: /Users/maura/Documents/COSI/data-challenge-4/workflow/data/config/dataio_continuum.yaml
Orientation: /Users/maura/Documents/COSI/data-challenge-4/workflow/data/inputs/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits
Continuum response: /Users/maura/Documents/COSI/data-challenge-4/workflow/data/inputs/ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5
Simulated background: /Users/maura/Documents/COSI/data-challenge-4/workflow/data/inputs/Total_DC4_BG_3months_unbinned_data_filtered_with_SAAcut_withSAAbck.fits.gz
Weekly files: 14
Output directory: /Users/maura/Documents/COSI/data-challenge-4/workflow/data/outputs


## Check input availability

Inspect the files before downloading or binning. The response is needed for the subsequent fit.

In [5]:
# Report each input and whether it is available locally.
required_inputs = {
    **{f"Week {week}": path for week, path in weekly_paths.items()},
    "Simulated background": background_path,
    "Orientation": orientation_path,
    "Continuum response": response_path,
    "Continuum configuration": config_path,
}
for label, path in required_inputs.items():
    print(f"{label}: {'found' if path.is_file() else 'MISSING'} — {path}")

Week 1: found — /Users/maura/Documents/COSI/data-challenge-4/workflow/data/inputs/dc4_mock_dataset_week_1_unbinned_data_filtered_with_SAAcut.fits.gz
Week 2: found — /Users/maura/Documents/COSI/data-challenge-4/workflow/data/inputs/dc4_mock_dataset_week_2_unbinned_data_filtered_with_SAAcut.fits.gz
Week 3: found — /Users/maura/Documents/COSI/data-challenge-4/workflow/data/inputs/dc4_mock_dataset_week_3_unbinned_data_filtered_with_SAAcut.fits.gz
Week 4: found — /Users/maura/Documents/COSI/data-challenge-4/workflow/data/inputs/dc4_mock_dataset_week_4_unbinned_data_filtered_with_SAAcut.fits.gz
Week 5: found — /Users/maura/Documents/COSI/data-challenge-4/workflow/data/inputs/dc4_mock_dataset_week_5_unbinned_data_filtered_with_SAAcut.fits.gz
Week 6: found — /Users/maura/Documents/COSI/data-challenge-4/workflow/data/inputs/dc4_mock_dataset_week_6_unbinned_data_filtered_with_SAAcut.fits.gz
Week 7: found — /Users/maura/Documents/COSI/data-challenge-4/workflow/data/inputs/dc4_mock_dataset_week_7_

## Download missing inputs

This cell downloads missing inputs from the project Wasabi locations. Downloads can be large and require repository access. The continuum configuration must exist locally.

In [6]:
# Download only files that do not already exist.
def download_if_missing(remote_path, local_path):
    if local_path.is_file():
        print(f"Already present: {local_path.name}")
        return
    print(f"Downloading: {local_path.name}")
    fetch_wasabi_file(remote_path, output=str(local_path))
    if not local_path.is_file():
        raise FileNotFoundError(f"Download did not create {local_path}")

# Obtain the observations, background, orientation and response.
paths.data_dir.mkdir(parents=True, exist_ok=True)
for week, local_path in weekly_paths.items():
    download_if_missing(paths.weekly_remote_paths[week], local_path)
download_if_missing(paths.simulated_background_remote_path, background_path)
download_if_missing(paths.orientation_remote_path, orientation_path)
download_if_missing(paths.response_remote_paths[1], response_path)
if not config_path.is_file():
    raise FileNotFoundError(f"Continuum configuration not found: {config_path}")

Already present: dc4_mock_dataset_week_1_unbinned_data_filtered_with_SAAcut.fits.gz
Already present: dc4_mock_dataset_week_2_unbinned_data_filtered_with_SAAcut.fits.gz
Already present: dc4_mock_dataset_week_3_unbinned_data_filtered_with_SAAcut.fits.gz
Already present: dc4_mock_dataset_week_4_unbinned_data_filtered_with_SAAcut.fits.gz
Already present: dc4_mock_dataset_week_5_unbinned_data_filtered_with_SAAcut.fits.gz
Already present: dc4_mock_dataset_week_6_unbinned_data_filtered_with_SAAcut.fits.gz
Already present: dc4_mock_dataset_week_7_unbinned_data_filtered_with_SAAcut.fits.gz
Already present: dc4_mock_dataset_week_8_unbinned_data_filtered_with_SAAcut.fits.gz
Already present: dc4_mock_dataset_week_9_unbinned_data_filtered_with_SAAcut.fits.gz
Already present: dc4_mock_dataset_week_10_unbinned_data_filtered_with_SAAcut.fits.gz
Already present: dc4_mock_dataset_week_11_unbinned_data_filtered_with_SAAcut.fits.gz
Already present: dc4_mock_dataset_week_12_unbinned_data_filtered_with_SAAc

## Inspect configuration and response

Compare the measured-energy and angular axes before binning. The incident-energy axis is needed for folding the source model.

In [7]:
# Display the binning configuration and response axes.
print(config_path.read_text())
with cp.response.FullDetectorResponse.open(response_path) as response:
    print("Response axes:", response.axes.labels)
    print(repr(response))

data_file: /Users/maura/Documents/COSI/data-challenge-4/workflow/data/inputs/dc4_mock_dataset_week_1_unbinned_data_filtered_with_SAAcut.fits.gz
ori_file: /Users/maura/Documents/COSI/data-challenge-4/workflow/data/inputs/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits
unbinned_output: fits
time_bins: 3600.0
energy_bins:
- 100.0
- 158.489
- 251.189
- 398.107
- 630.957
- 1000.0
- 1584.89
- 2511.89
- 3981.07
- 6309.57
- 10000.0
phi_pix_size: 6
nside: 8
scheme: RING
tmin: 1835487300.036206
tmax: 1843467254.887687

Response axes: ['NuLambda' 'Ei' 'Em' 'Phi' 'PsiChi']
FILENAME: '/Users/maura/Documents/COSI/data-challenge-4/workflow/data/inputs/ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5'
AXES:
  NuLambda:
    DESCRIPTION: 'Location of the simulated source in the spacecraft coordinates'
    TYPE: 'healpix'
    NPIX: 768
    NSIDE: 8
    SCHEME: 'RING'
  Ei:
    DESCRIPTION: 'Initial simulated energy'
    TYPE

## Bin weekly observations

Write one time-resolved multienergy histogram per week. Existing files are skipped; verify their provenance before reuse, especially after changing the configuration. This step can take substantial time and disk space.

In [8]:
# Bin each week without overwriting existing products.
output_dir.mkdir(parents=True, exist_ok=True)
for week, input_path in weekly_paths.items():
    output_path = weekly_binned_paths[week]
    if output_path.exists():
        print(f"Week {week}: existing output, skipped: {output_path}")
        continue
    print(f"Week {week}: binning {input_path.name}")
    analysis = cp.BinnedData(config_path)
    analysis.get_binned_data(
        unbinned_data=str(input_path),
        output_name=str(output_path.with_suffix("")),
    )
    print(f"Week {week}: saved to {output_path}")

Week 1: existing output, skipped: /Users/maura/Documents/COSI/data-challenge-4/workflow/data/outputs/dc4_week_1_crab_continuum_binned.hdf5
Week 2: existing output, skipped: /Users/maura/Documents/COSI/data-challenge-4/workflow/data/outputs/dc4_week_2_crab_continuum_binned.hdf5
Week 3: existing output, skipped: /Users/maura/Documents/COSI/data-challenge-4/workflow/data/outputs/dc4_week_3_crab_continuum_binned.hdf5
Week 4: existing output, skipped: /Users/maura/Documents/COSI/data-challenge-4/workflow/data/outputs/dc4_week_4_crab_continuum_binned.hdf5
Week 5: existing output, skipped: /Users/maura/Documents/COSI/data-challenge-4/workflow/data/outputs/dc4_week_5_crab_continuum_binned.hdf5
Week 6: existing output, skipped: /Users/maura/Documents/COSI/data-challenge-4/workflow/data/outputs/dc4_week_6_crab_continuum_binned.hdf5
Week 7: existing output, skipped: /Users/maura/Documents/COSI/data-challenge-4/workflow/data/outputs/dc4_week_7_crab_continuum_binned.hdf5
Week 8: existing output, sk

## Bin the simulated background

Use the same configuration for the three-month background simulation. This output is not a fitted background model. Its normalization and time alignment remain to be checked.

In [9]:
# Bin the simulation without overwriting an existing output.
if background_binned_path.exists():
    print("Existing background output, skipped:", background_binned_path)
else:
    background_analysis = cp.BinnedData(config_path)
    background_analysis.get_binned_data(
        unbinned_data=str(background_path),
        output_name=str(background_binned_path.with_suffix("")),
    )
    print("Saved background:", background_binned_path)

Existing background output, skipped: /Users/maura/Documents/COSI/data-challenge-4/workflow/data/outputs/simulated_background_crab_continuum_binned.hdf5


## Inspect prepared histograms

Inspect one weekly histogram and the background without assuming their time axes match.

In [10]:
# Load one representative observation and the simulated background.
example_week = 1
example_analysis = cp.BinnedData(config_path)
example_analysis.load_binned_data_from_hdf5(binned_data=str(weekly_binned_paths[example_week]))
background_analysis = cp.BinnedData(config_path)
background_analysis.load_binned_data_from_hdf5(binned_data=str(background_binned_path))

# Inspect axes without converting the histograms to dense arrays.
for label, histogram in [
    (f"Observation week {example_week}", example_analysis.binned_data),
    ("Simulated background", background_analysis.binned_data),
]:
    print(f"\n{label}")
    print("Axes:", histogram.axes.labels)
    print("Shape:", histogram.shape)
    print("Measured-energy edges:", histogram.axes["Em"].edges)
    print("Time edges:", histogram.axes["Time"].edges)

# Compare the non-temporal axes used for residual analysis.
for axis_name in ("Em", "Phi", "PsiChi"):
    matches = example_analysis.binned_data.axes[axis_name] == background_analysis.binned_data.axes[axis_name]
    print(f"{axis_name} axes match: {matches}")


Observation week 1
Axes: ['Time' 'Em' 'Phi' 'PsiChi']
Shape: (2217, 10, 30, 768)
Measured-energy edges: [  100.      158.489   251.189   398.107   630.957  1000.     1584.89
  2511.89   3981.07   6309.57  10000.   ] keV
Time edges: [1.83548730e+09 1.83549090e+09 1.83549450e+09 ... 1.84346006e+09
 1.84346366e+09 1.84346725e+09] s

Simulated background
Axes: ['Time' 'Em' 'Phi' 'PsiChi']
Shape: (2217, 10, 30, 768)
Measured-energy edges: [  100.      158.489   251.189   398.107   630.957  1000.     1584.89
  2511.89   3981.07   6309.57  10000.   ] keV
Time edges: [1.83548730e+09 1.83549090e+09 1.83549450e+09 ... 1.84346006e+09
 1.84346366e+09 1.84346725e+09] s
Em axes match: True
Phi axes match: True
PsiChi axes match: True


In [11]:
# Compare the complete time axes of the observation and background.
time_axes_match = (
    example_analysis.binned_data.axes["Time"]
    == background_analysis.binned_data.axes["Time"]
)
print("Time axes match:", time_axes_match)

Time axes match: True


## Validate all weekly histogram axes

Load each weekly product separately and compare its complete axes and shape to the background. Matching axes establish binning compatibility; they do not prove input provenance or identical event selection. Existing files skipped during binning still need provenance checks.

In [12]:
# Compare each weekly product with the time-resolved background.
reference_histogram = background_analysis.binned_data
axis_labels = ("Time", "Em", "Phi", "PsiChi")
weekly_axes_match = {}

for week, output_path in sorted(weekly_binned_paths.items()):
    weekly_analysis = cp.BinnedData(config_path)
    weekly_analysis.load_binned_data_from_hdf5(binned_data=str(output_path))
    weekly_histogram = weekly_analysis.binned_data
    axis_matches = {
        label: weekly_histogram.axes[label] == reference_histogram.axes[label]
        for label in axis_labels
    }
    shape_matches = weekly_histogram.shape == reference_histogram.shape
    weekly_axes_match[week] = shape_matches and all(axis_matches.values())
    print(f"Week {week}: shape={shape_matches}, axes={axis_matches}")

print("All weekly axes and shapes match:", all(weekly_axes_match.values()))

Week 1: shape=True, axes={'Time': np.True_, 'Em': np.True_, 'Phi': np.True_, 'PsiChi': True}
Week 2: shape=True, axes={'Time': np.True_, 'Em': np.True_, 'Phi': np.True_, 'PsiChi': True}
Week 3: shape=True, axes={'Time': np.True_, 'Em': np.True_, 'Phi': np.True_, 'PsiChi': True}
Week 4: shape=True, axes={'Time': np.True_, 'Em': np.True_, 'Phi': np.True_, 'PsiChi': True}
Week 5: shape=True, axes={'Time': np.True_, 'Em': np.True_, 'Phi': np.True_, 'PsiChi': True}
Week 6: shape=True, axes={'Time': np.True_, 'Em': np.True_, 'Phi': np.True_, 'PsiChi': True}
Week 7: shape=True, axes={'Time': np.True_, 'Em': np.True_, 'Phi': np.True_, 'PsiChi': True}
Week 8: shape=True, axes={'Time': np.True_, 'Em': np.True_, 'Phi': np.True_, 'PsiChi': True}
Week 9: shape=True, axes={'Time': np.True_, 'Em': np.True_, 'Phi': np.True_, 'PsiChi': True}
Week 10: shape=True, axes={'Time': np.True_, 'Em': np.True_, 'Phi': np.True_, 'PsiChi': True}
Week 11: shape=True, axes={'Time': np.True_, 'Em': np.True_, 'Phi': n

## Compare the detector response axes

Compare measured-energy and Compton-angle axes directly. The response PsiChi axis is in spacecraft coordinates, while the binned data use Galactic coordinates. Inspect their HEALPix resolution and ordering separately; matching these does not establish coordinate compatibility. The folded prediction must be checked against the data after applying the observation orientation.


In [13]:
# Compare the measured-energy and Compton-angle axes.
with cp.response.FullDetectorResponse.open(response_path) as response:
    for label in ("Em", "Phi"):
        print(f"{label} response and background axes match:",
              response.axes[label] == reference_histogram.axes[label])

    # Compare the HEALPix geometry and report both coordinate frames.
    response_axis = response.axes["PsiChi"]
    background_axis = reference_histogram.axes["PsiChi"]
    print("PsiChi NSIDE matches:", response_axis.nside == background_axis.nside)
    print("PsiChi scheme matches:", response_axis.scheme == background_axis.scheme)
    print("Response coordinate system:", response_axis.coordsys)
    print("Background coordinate system:", background_axis.coordsys)


Em response and background axes match: True
Phi response and background axes match: True
PsiChi NSIDE matches: True
PsiChi scheme matches: True
Response coordinate system: <SpacecraftFrame Frame (attitude=None, obstime=None, location=None)>
Background coordinate system: <Galactic Frame>


In [14]:
# Inspect the background and detector-response PsiChi axes.
background_psichi = reference_histogram.axes["PsiChi"]

with cp.response.FullDetectorResponse.open(response_path) as response:
    response_psichi = response.axes["PsiChi"]

    for name, axis in [
        ("Background", background_psichi),
        ("Detector response", response_psichi),
    ]:
        print(f"\n{name}:")
        print("  NSIDE:", axis.nside)
        print("  Scheme:", axis.scheme)
        print("  Coordinate system:", axis.coordsys)


Background:
  NSIDE: 8
  Scheme: RING
  Coordinate system: <Galactic Frame>

Detector response:
  NSIDE: 8
  Scheme: RING
  Coordinate system: <SpacecraftFrame Frame (attitude=None, obstime=None, location=None)>


## Investigate background events outside numerical axis limits

The earlier fill warning does not identify an axis or the number of discarded events. The cell below reads the original compressed background FITS and counts values outside Time, Em and Phi bounds, including non-finite values. It uses chunks for the comparisons, but reading the compressed FITS can still require substantial time and memory. It does not rebin or modify files.

Counts per axis can overlap. The union counts an event once across these three axes; it does not test the PsiChi coordinate or establish a complete accounting of all possible rejections. The installed COSIpy binning code uses all rows except the last by default, and this diagnostic follows that same selection. Verify the bin-edge convention before interpreting boundary counts.

In [15]:
# Read the numerical axis limits from the histogram that was actually saved.
time_limits = reference_histogram.axes["Time"].edges.to_value(u.s)[[0, -1]]
energy_limits = reference_histogram.axes["Em"].edges.to_value(u.keV)[[0, -1]]
phi_limits = reference_histogram.axes["Phi"].edges.to_value(u.deg)[[0, -1]]

# Count invalid values and values outside the half-open numerical intervals.
axis_counts = {label: 0 for label in ("Time", "Em", "Phi")}
outside_any_numerical_axis = 0
chunk_size = 1_000_000

with fits.open(background_path, memmap=True) as hdul:
    events = hdul[1].data
    # Match COSIpy's default [0:-1] event selection.
    selected_rows = max(len(events) - 1, 0)
    print("Total input rows:", len(events))
    print("Rows included in the binning selection:", selected_rows)

    for start in range(0, selected_rows, chunk_size):
        stop = min(start + chunk_size, selected_rows)
        chunk = events[start:stop]
        values_by_axis = {
            "Time": (np.asarray(chunk["TimeTags"]), time_limits),
            "Em": (np.asarray(chunk["Energies"]), energy_limits),
            "Phi": (np.rad2deg(np.asarray(chunk["Phi"])), phi_limits),
        }
        outside_any = np.zeros(stop - start, dtype=bool)
        for label, (values, limits) in values_by_axis.items():
            outside = ~np.isfinite(values) | (values < limits[0]) | (values >= limits[1])
            axis_counts[label] += int(np.count_nonzero(outside))
            outside_any |= outside
        outside_any_numerical_axis += int(np.count_nonzero(outside_any))

print("Outside limits or non-finite, by axis:", axis_counts)
print("Outside at least one numerical axis:", outside_any_numerical_axis)
print("Numerical axis limits — Time [s]:", time_limits)
print("Numerical axis limits — Em [keV]:", energy_limits)
print("Numerical axis limits — Phi [deg]:", phi_limits)

Total input rows: 168648544
Rows included in the binning selection: 168648543
Outside limits or non-finite, by axis: {'Time': 1, 'Em': 0, 'Phi': 0}
Outside at least one numerical axis: 1
Numerical axis limits — Time [s]: [1.83548730e+09 1.84346725e+09]
Numerical axis limits — Em [keV]: [  100. 10000.]
Numerical axis limits — Phi [deg]: [  0. 180.]
